# Paul Graham essay feeds

Unofficial **RSS · Atom · JSON Feed** for
[paulgraham.com/articles.html](https://paulgraham.com/articles.html).

Live fetch → parse → validate → write feeds on this runtime, then download
`feeds.zip`.

### How to run

1. Set **Options** below  
2. **Runtime → Run all**  
3. Save **`feeds.zip`** when the browser download starts  

> [!NOTE]
> The generate cell uses `uvx --from git+…` against the repo default branch (`main`). That float can change under you. For reproducible Colab runs, pin to a **release tag** or **commit SHA**, e.g. `git+https://github.com/wyattowalsh/paul-graham-essay-feeds@vX.Y.Z` or `…@abcdef1` — do not assume a fixed `@v0.1.0` until you choose one.

> Not affiliated with or endorsed by Paul Graham.


In [ ]:
#@title Options { display-mode: "form" }
#@markdown ### Output
#@markdown Directory for generated `feeds/` (Colab default is fine).
ROOT = "/content/pg-feeds"  #@param {type:"string"}
#@markdown ### Pipeline
#@markdown **Cost:** ~1 GET per essay page. Uncheck for index-only / faster runs.
ENRICH = True  #@param {type:"boolean"}
#@markdown Live HEAD/GET every essay URL (extra GETs; slowest).
VALIDATE_LINKS = False  #@param {type:"boolean"}


In [ ]:
#@title 1 · Install tools & logging { display-mode: "form" }
#@markdown Installs `uv` (for `uvx`) and `loguru` for notebook status lines.

"""Bootstrap the Colab kernel toolchain.

The production pipeline still runs through the published CLI (`uvx … pg-essay-feeds`).
This cell only prepares the runner and a clean loguru sink for notebook messages.
"""

from __future__ import annotations

# Colab shell: quiet install of CLI runner + logger used below.
!pip install -q uv loguru

from loguru import logger

# Avoid duplicate handlers when re-running the cell.
logger.remove()
logger.add(
    lambda msg: print(msg, end=""),
    format="<green>{time:HH:mm:ss}</green> | <level>{level:<7}</level> | {message}\n",
    level="INFO",
    colorize=True,
)

logger.info("Toolchain ready (uv + loguru)")
logger.info("Configured output root: {}", ROOT)


In [ ]:
#@title 2 · Generate live feeds { display-mode: "form" }
#@markdown Fetches the **live** essays index, builds RSS/Atom/JSON under `ROOT`, then runs `check`.

"""Live generation (not a copy of repo artifacts).

`uvx --from git+…` installs the latest package ephemerally and runs:
  pg-essay-feeds update  →  fetch / extract / enrich / validate / write
  pg-essay-feeds check   →  structural sanity on the outputs
"""

from __future__ import annotations

from pathlib import Path

from loguru import logger

# Resolve and create the runtime workspace.
root = Path(ROOT).expanduser().resolve()
root.mkdir(parents=True, exist_ok=True)
feeds_dir = root / "feeds"

# Published package source — always network-install the CLI, never clone+copy feeds/.
# Floating main: prefer pinning @vX.Y.Z or @<commit> for reproducible runs (see intro note).
PKG = "git+https://github.com/wyattowalsh/paul-graham-essay-feeds"

logger.info("Generating feeds under {}", root)

# Branch on form toggles so `!` never receives empty flag tokens.
if ENRICH and VALIDATE_LINKS:
    logger.info("Mode: enrich + validate-links")
    !uvx --from {PKG} pg-essay-feeds update --repo-root {root} --validate-links
elif ENRICH:
    logger.info("Mode: enrich (default)")
    !uvx --from {PKG} pg-essay-feeds update --repo-root {root}
elif VALIDATE_LINKS:
    logger.info("Mode: no-enrich + validate-links")
    !uvx --from {PKG} pg-essay-feeds update --repo-root {root} --no-enrich --validate-links
else:
    logger.info("Mode: index only (--no-enrich)")
    !uvx --from {PKG} pg-essay-feeds update --repo-root {root} --no-enrich

logger.info("Running structural check…")
!uvx --from {PKG} pg-essay-feeds check --repo-root {root}

# Fail loud if expected artifacts are missing or empty.
required = ("rss.xml", "atom.xml", "feed.json")
for name in required:
    path = feeds_dir / name
    if not path.is_file() or path.stat().st_size == 0:
        raise FileNotFoundError(f"Expected non-empty feed file missing: {path}")
    logger.info("  {:12} {:>8} bytes", name, path.stat().st_size)

logger.success("Generation complete → {}", feeds_dir)


In [ ]:
#@title 3 · Download feeds.zip { display-mode: "form" }
#@markdown Zips RSS / Atom / JSON Feed and starts a browser download on Colab.

"""Package the three feed files for offline use.

Requires the generate cell to have succeeded first.
"""

from __future__ import annotations

import zipfile
from pathlib import Path

from loguru import logger

root = Path(ROOT).expanduser().resolve()
feeds_dir = root / "feeds"
zip_path = root / "feeds.zip"

members = [feeds_dir / name for name in ("rss.xml", "atom.xml", "feed.json")]
missing = [p for p in members if not p.is_file()]
if missing:
    raise FileNotFoundError(
        "Run «2 · Generate live feeds» first. Missing: "
        + ", ".join(str(p) for p in missing)
    )

if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in members:
        zf.write(path, arcname=path.name)
        logger.info("zip + {} ({} bytes)", path.name, path.stat().st_size)

logger.success("Wrote {}", zip_path)

try:
    from google.colab import files

    files.download(str(zip_path))
    logger.info("Browser download started")
except ImportError:
    logger.warning("Not running in Colab — open the zip at {}", zip_path)
